# Structuring Your Unstructured Data with AI SQL Functions

Most real-world data isn't clean rows and columns — it's free-form text, PDFs, and documents. In this notebook we'll use Databricks **AI SQL Functions** to turn that unstructured data into structured, queryable tables using a single SQL expression.

**Why batch inference matters:**
- **Analysts** get structured columns they can filter, aggregate, and visualize
- **Classical ML** gets clean features without manual labelling pipelines
- **Agentic systems** get structured memory and tool-ready outputs

**What we'll cover:**
1. AI SQL Functions overview
2. Structuring free-form text (`ai_classify`, `ai_extract`, `ai_query`)
3. Wrapping a custom prompt in a reusable Unity Catalog function
4. Parsing documents (`ai_parse` on a PDF)
5. Evaluating AI function outputs
6. Challenge activity

In [ ]:
catalog_to_write_to = "databricks_day"
schema_to_write_to  = "weather"

write_path  = f"{catalog_to_write_to}.{schema_to_write_to}"
volume_path = f"/Volumes/{catalog_to_write_to}/{schema_to_write_to}/files"

source_table = "accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric"

## 1. AI SQL Functions Overview

Databricks AI SQL Functions are built-in SQL functions that call foundation models directly inside a query. There's no model deployment, no API key management, and no separate inference pipeline — the model call happens inline, one row at a time, as part of your SQL or PySpark expression.

Under the hood they use the **Databricks Foundation Model API**, which is served through Unity Catalog and governed like any other data asset.

| Function | What it does |
|---|---|
| `ai_classify(text, labels)` | Assigns one label from a fixed list to each row |
| `ai_extract(text, fields)` | Pulls named attributes out of free-form text as a struct |
| `ai_query(model, prompt)` | Calls a model with a custom prompt — most flexible |
| `ai_parse(content, format)` | Extracts structured text from binary files (PDFs, images) |

**Key properties:**
- Works in SQL, PySpark (`expr()`), and Databricks notebooks
- Scales to millions of rows via Spark parallelism
- Results can be written straight to a Delta table — making them available to dashboards, Genie, and downstream pipelines

## 2. Structuring Free-Form Text

The AccuWeather dataset has a `phrase_long` column containing natural-language weather descriptions like `"Mostly cloudy with a few showers"` or `"Sunny and pleasant"`. These are useful to read but hard to filter or aggregate.

We'll apply AI SQL functions to turn them into structured columns — without writing a single line of ML code.

First, let's load a deduplicated sample of phrases into a temp view so we can query it with SQL.

In [ ]:
df_phrases = (
    spark.table(source_table)
    .select("station_code", "date", "phrase_long")
    .dropna(subset=["phrase_long"])
    .dropDuplicates(["phrase_long"])
    .limit(50)
)

df_phrases.createOrReplaceTempView("weather_phrases")

display(df_phrases)

### 2a. `ai_classify` — Assign a label from a fixed list

`ai_classify` picks the single best-matching label from the list you provide. It's deterministic and fast — ideal for categorisation tasks where you control the label set.

In [ ]:
%sql
SELECT
    phrase_long,
    ai_classify(phrase_long, ARRAY('Clear', 'Cloudy', 'Precipitation', 'Severe')) AS weather_category
FROM weather_phrases
ORDER BY weather_category
LIMIT 20

### 2b. `ai_extract` — Pull named attributes from text

`ai_extract` reads free-form text and returns a struct of named fields. You define what you want to extract — the model figures out how to find it in the text.

In [ ]:
%sql
SELECT
    phrase_long,
    ai_extract(
        phrase_long,
        ARRAY('precipitation_type', 'sky_condition', 'intensity')
    ) AS extracted
FROM weather_phrases
LIMIT 20

You can flatten the struct into individual columns using dot notation or `.*`:

```sql
SELECT phrase_long, extracted.precipitation_type, extracted.sky_condition
FROM (...)
```

### 2c. `ai_query` — Custom prompt, maximum flexibility

`ai_query` gives you a direct line to the model. You pass the model name and a full prompt string — useful when classify and extract don't cover your use case.

In [ ]:
%sql
SELECT
    phrase_long,
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'Is this weather condition suitable for flying a small aircraft? ',
            'Answer Yes or No, then give a one-sentence reason. ',
            'Condition: ', phrase_long
        )
    ) AS flying_suitability
FROM weather_phrases
LIMIT 10

## 3. Wrapping a Prompt in a Reusable Unity Catalog Function

Repeating a long prompt string in every notebook is fragile — if the prompt changes you have to find and update every reference. A better pattern is to wrap the `ai_query` call in a **Python function registered to Unity Catalog**.

Benefits:
- **Reusable** — call it from any notebook, SQL editor, or job with `catalog.schema.function_name()`
- **Governed** — versioned and permissioned like any other UC asset
- **Consistent** — the prompt logic lives in one place

The function below analyses a weather phrase and returns a **structured JSON string** with three fields: `suitable` (bool), `risk_level` (High / Medium / Low), and `reason` (one sentence). Returning JSON rather than plain text makes it easy to parse the output into columns downstream.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {write_path}.assess_weather_for_flight(phrase STRING)
RETURNS STRING
COMMENT 'Assesses a weather condition phrase for small-aircraft flight suitability. Returns JSON with keys: suitable (boolean), risk_level (High/Medium/Low), reason (string).'
LANGUAGE PYTHON
AS $$
import json

def assess_weather_for_flight(phrase):
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

    prompt = (
        "You are an aviation weather analyst. Given a weather condition description, "
        "assess whether it is suitable for flying a small general aviation aircraft. "
        "Respond ONLY with a JSON object using these exact keys: "
        "\\\"suitable\\\" (boolean), "
        "\\\"risk_level\\\" (one of: High, Medium, Low), "
        "\\\"reason\\\" (one sentence). "
        f"Weather condition: {phrase}"
    )

    response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[{{"role": "user", "content": prompt}}]
    )

    return response.choices[0].message.content

return assess_weather_for_flight(phrase)
$$
""")

**Call the registered UC function from SQL**

Once registered, anyone with `EXECUTE` permission on the function can call it by its three-part name.

In [ ]:
%sql
SELECT
    phrase_long,
    databricks_day.weather.assess_weather_for_flight(phrase_long) AS assessment_json
FROM weather_phrases
LIMIT 10

**Parse the JSON output into typed columns**

Since the function returns a JSON string, we can use `from_json` to explode it into proper columns — making it filterable and aggregatable like any other structured data.

In [ ]:
from pyspark.sql.functions import col, expr, from_json
from pyspark.sql.types import StructType, StructField, BooleanType, StringType

assessment_schema = StructType([
    StructField("suitable",   BooleanType(), True),
    StructField("risk_level", StringType(),  True),
    StructField("reason",     StringType(),  True),
])

df_assessed = spark.sql(f"""
    SELECT
        phrase_long,
        {write_path}.assess_weather_for_flight(phrase_long) AS assessment_json
    FROM weather_phrases
    LIMIT 20
""")

df_structured = df_assessed.withColumn(
    "assessment",
    from_json(col("assessment_json"), assessment_schema)
).select(
    "phrase_long",
    col("assessment.suitable").alias("suitable"),
    col("assessment.risk_level").alias("risk_level"),
    col("assessment.reason").alias("reason"),
)

display(df_structured)

## 4. Parsing Documents with `ai_parse`

So far we've worked with short text strings. But a lot of valuable information lives in documents — PDFs, reports, manuals. `ai_parse` reads a binary file from a Databricks Volume and extracts its content as structured text (markdown, plain text, or JSON).

We'll use the **RCAF Weather Manual** — a document describing aviation weather standards — as our example. The goal is to:
1. Read the PDF from the Volume as a binary file
2. Parse it with `ai_parse` to extract readable text
3. Write the parsed content to a Delta table
4. Query that table with `ai_query` to ask questions of the document

**Step 1: Read the PDF as a binary file**

`spark.read.format("binaryFile")` reads files from a Volume path into a DataFrame where the `content` column holds the raw bytes.

In [ ]:
df_pdf = spark.read.format("binaryFile").load(f"{volume_path}/RCAF-Weather-Manual.pdf")

display(df_pdf.select("path", "length", "modificationTime"))

**Step 2: Parse with `ai_parse`**

`ai_parse(content, 'markdown')` converts the binary PDF into markdown-formatted text, preserving headings, tables, and structure where possible.

In [ ]:
from pyspark.sql.functions import expr

df_parsed = df_pdf.select(
    "path",
    "modificationTime",
    expr("ai_parse(content, 'markdown')").alias("parsed_content")
)

display(df_parsed)

**Step 3: Write the parsed output to a Delta table**

Persisting the parsed text means you only pay to parse the PDF once. Downstream queries run against the Delta table, not the binary file.

In [ ]:
df_parsed.write.mode("overwrite").saveAsTable(f"{write_path}.rcaf_manual_parsed")

print("Rows written:", spark.table(f"{write_path}.rcaf_manual_parsed").count())

**Step 4: Query the document with `ai_query`**

Now that the parsed text is in a Delta table, we can use `ai_query` to ask natural-language questions directly against it.

In [ ]:
%sql
SELECT
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'You are reading an aviation weather manual. Answer concisely based only on the content provided. ',
            'Question: What ceiling and visibility minimums are required for VFR flight? ',
            'Manual content: ', parsed_content
        )
    ) AS vfr_minimums
FROM databricks_day.weather.rcaf_manual_parsed

You can change the question in the prompt without touching the parsing step — the Delta table is the durable source. Try asking:
- *"What weather phenomena are considered hazardous to flight?"*
- *"How is icing classified in this manual?"*
- *"Summarise the key points about turbulence in 3 bullet points."*

## 5. Evaluating AI Function Outputs

AI SQL functions are powerful, but their outputs are probabilistic — the same input can produce slightly different results, and the model can make mistakes. Before trusting batch inference results in a production pipeline, it's worth thinking about evaluation.

**Three-step approach:**

**1. Create a labelled benchmark**
Hand-label 50–100 rows with the correct answer. This becomes your ground truth. Keep it small enough to do manually, but large enough to be statistically meaningful. Store it as a Delta table.

**2. Run your AI function against the benchmark**
Apply `ai_classify`, `ai_extract`, or your UDF to the benchmark rows and compare the outputs to your labels. Calculate accuracy, precision, recall, or whatever metric fits your use case.

**3. Use `ai_query` as an LLM judge**
For tasks where there's no single right answer (e.g. free-text generation), use a second `ai_query` call to evaluate the quality of the first. The judge prompt asks the model to score or compare outputs. This is called **LLM-as-a-judge**.

```sql
-- Example: use ai_query to judge whether a classification was correct
SELECT
    phrase_long,
    predicted_category,
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'A weather classifier labelled the phrase "', phrase_long, '" as "', predicted_category, '". ',
            'Is this label correct? Answer only: Correct or Incorrect.'
        )
    ) AS judge_verdict
FROM benchmark_results
```

**Practical tips:**
- Re-evaluate after prompt changes — small wording changes can shift accuracy meaningfully
- Track agreement rate (% where the judge agrees with your label) over time as a quality signal
- For production pipelines, fail the job or alert if accuracy drops below a threshold

---

## 6. Challenge Activity

**Goal:** Build a structured weather conditions table from the AccuWeather `phrase_long` data.

You're going to create a table called `weather_conditions_structured` that enriches the raw weather phrases with AI-generated labels and attributes — ready to use in dashboards, Genie, or downstream pipelines.

### Your pipeline should:
1. **Load** a sample of weather phrases from the source table
2. **Classify** each phrase as `"Good Flying"` or `"Poor Flying"` using `ai_classify`
3. **Extract** `precipitation_type` and `sky_condition` from each phrase using `ai_extract`
4. **Combine** the classification and extracted fields into a single DataFrame
5. **Persist** the result to `weather_conditions_structured`

### Stretch goal:
- Call your registered UC function `assess_weather_for_flight` and parse its JSON output into typed columns (`suitable`, `risk_level`, `reason`)
- Add these columns to your table using a Delta merge

**Hints are in the cells below — fill in the `# TODO` sections.**

**Step 1 & 2: Load phrases and classify**

In [ ]:
# TODO: Load a sample of distinct phrase_long values from the source table into a temp view
# Hint: similar to how we built weather_phrases at the start of Section 2

df_challenge = # TODO

df_challenge.createOrReplaceTempView("challenge_phrases")

In [ ]:
%sql
-- TODO: Use ai_classify to label each phrase as "Good Flying" or "Poor Flying"
-- Hint: ai_classify(column, ARRAY('label1', 'label2'))

SELECT
    phrase_long,
    -- TODO
FROM challenge_phrases
LIMIT 20

**Step 3 & 4: Extract attributes and combine into a single DataFrame**

In [ ]:
df_conditions = spark.sql("""
    SELECT
        phrase_long,
        -- TODO: classify as Good Flying / Poor Flying using ai_classify
        -- TODO: extract precipitation_type and sky_condition using ai_extract
    FROM challenge_phrases
    LIMIT 30
""")

display(df_conditions)

**Step 5: Persist to a new Delta table**

In [ ]:
# TODO: Write df_conditions to a new table called "weather_conditions_structured"
# Hint: df.write.mode("overwrite").saveAsTable(f"{write_path}.table_name")

# TODO

print("Rows written:", spark.table(f"{write_path}.weather_conditions_structured").count())

---

### Stretch Goal: Call the UC function and parse JSON into typed columns

Use the `assess_weather_for_flight` function you registered to Unity Catalog, then expand its JSON output into `suitable`, `risk_level`, and `reason` columns. Merge the result back into your table.

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, BooleanType, StringType

assessment_schema = StructType([
    StructField("suitable",   BooleanType(), True),
    StructField("risk_level", StringType(),  True),
    StructField("reason",     StringType(),  True),
])

# TODO: Call assess_weather_for_flight on the challenge phrases and parse the JSON output
# Hint: refer to Section 3 where we did this — use from_json() and col("assessment.*")

df_assessed = spark.sql("""
    SELECT
        phrase_long,
        -- TODO: call databricks_day.weather.assess_weather_for_flight(phrase_long) AS assessment_json
    FROM challenge_phrases
    LIMIT 20
""")

# TODO: parse assessment_json using from_json and expand into columns

display(df_assessed)

In [ ]:
from delta.tables import DeltaTable

# TODO: Merge df_assessed into weather_conditions_structured, matching on phrase_long
# Hint: refer back to the Delta merge pattern in notebook 1.0, Section 3

# TODO